In [1]:
%pip install tensorflow
%pip install torch
%pip install torchvision timm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
import matplotlib.pyplot as plt
import os

print(os.listdir("art"))

# Innstillingar
DATA_DIR = "art"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

# Last inn trenings- og valideringsdata
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
print("Klassar:", class_names)

# Gjer dataflyten raskare
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Pretrained base model
base_model = EfficientNetB0(
    include_top=False,
    weights=None,
    input_shape=(224, 224, 3)
)

base_model.trainable = True  # frys først

# Bygg modellen
inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(3, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Tren modellen
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS
)

# Lagre modellen
os.makedirs("models", exist_ok=True)
model.save("models/ai_vs_real_model.keras")

# Plot accuracy
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.legend()
plt.title("Training accuracy")
plt.savefig("results_accuracy.png")
plt.show()

['.DS_Store', 'real', 'ai']
Found 962 files belonging to 2 classes.
Using 770 files for training.
Found 962 files belonging to 2 classes.
Using 192 files for validation.
Klassar: ['ai', 'real']
Epoch 1/10


ValueError: Arguments `target` and `output` must have the same rank (ndim). Received: target.shape=(None,), output.shape=(None, 3)

In [3]:
#community forensics





import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm

# Load pretrained backbone
model = timm.create_model("vit_base_patch16_384", pretrained=True, num_classes=1)

# Freeze backbone, only train head (faster, less data needed)
for param in model.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True

# Dataset (folder structure: /data/real/ and /data/fake/)
transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder("./art", transform=transform)
print("Klasser:", dataset.classes)        # ['ai', 'real']
print("Antall bilder:", len(dataset))
print("Klasse → indeks:", dataset.class_to_idx)  # {'ai': 0, 'real': 1}

loader = DataLoader(dataset, batch_size=16, shuffle=True)


# Train
optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(5):
    for imgs, labels in loader:
        labels = labels.float().unsqueeze(1)
        preds = model(imgs)
        loss = criterion(preds, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} done")

torch.save(model.state_dict(), "finetuned_detector.pth")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Klasser: ['ai', 'real']
Antall bilder: 951
Klasse → indeks: {'ai': 0, 'real': 1}


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
